In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import random

from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

PROJECT_PATH = "/content/drive/MyDrive/AML_Project"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
customers = pd.read_csv(f"{PROJECT_PATH}/dataset/customers.csv")
cities = pd.read_csv(f"{PROJECT_PATH}/dataset/cities.csv")
atms = pd.read_csv(f"{PROJECT_PATH}/dataset/atms.csv")
merchants = pd.read_csv(f"{PROJECT_PATH}/dataset/merchants.csv")

print("Customers :",len(customers))
print("Cities :",len(cities))
print("ATMs :",len(atms))
print("Merchants :",len(merchants))

Customers : 5000
Cities : 20
ATMs : 200
Merchants : 501


In [ ]:
transaction_profiles = {

    "Student":{
        "UPI":35,
        "Food":30,
        "Shopping":20,
        "Recharge":15
    },

    "Engineer":{
        "UPI":20,
        "Fuel":25,
        "Shopping":25,
        "Card":20,
        "Transfer":10
    },

    "Doctor":{
        "Transfer":35,
        "Investment":20,
        "Card":25,
        "Healthcare":20
    },

    "Teacher":{
        "UPI":35,
        "Bills":30,
        "ATM":20,
        "Shopping":15
    },

    "Business":{
        "Transfer":40,
        "Salary":20,
        "NEFT":15,
        "RTGS":15,
        "Cash Deposit":10
    },

    "Government Employee":{
        "Bills":35,
        "ATM":30,
        "UPI":20,
        "Shopping":15
    },

    "Lawyer":{
        "Transfer":45,
        "Card":30,
        "Shopping":25
    },

    "Freelancer":{
        "UPI":35,
        "Transfer":35,
        "Shopping":30
    },

    "Retired":{
        "ATM":40,
        "Bills":35,
        "Healthcare":25
    },

    "Shop Owner":{
        "Transfer":35,
        "Cash Deposit":30,
        "UPI":20,
        "Shopping":15
    }

}

In [ ]:
def generate_amount(occupation, tx_type):

    if tx_type == "Salary":
        return random.randint(30000, 120000)

    elif tx_type == "Shopping":
        return random.randint(500, 15000)

    elif tx_type == "Food":
        return random.randint(100, 2000)

    elif tx_type == "Fuel":
        return random.randint(500, 5000)

    elif tx_type == "Bills":
        return random.randint(500, 10000)

    elif tx_type == "Healthcare":
        return random.randint(1000, 25000)

    elif tx_type == "ATM":
        return random.randint(500, 10000)

    elif tx_type == "Recharge":
        return random.randint(100, 1000)

    elif tx_type == "Investment":
        return random.randint(5000, 100000)

    elif tx_type == "Transfer":
        return random.randint(1000, 50000)

    elif tx_type == "NEFT":
        return random.randint(5000, 100000)

    elif tx_type == "RTGS":
        return random.randint(200000, 1000000)

    elif tx_type == "Cash Deposit":
        return random.randint(5000, 100000)

    elif tx_type == "Card":
        return random.randint(500, 20000)

    else:
        return random.randint(100, 5000)

In [ ]:
START_DATE = datetime(2026,1,1)

def generate_timestamp(day):

    base = START_DATE + timedelta(days=day)

    hour = random.randint(7,22)

    minute = random.randint(0,59)

    second = random.randint(0,59)

    return datetime(
        base.year,
        base.month,
        base.day,
        hour,
        minute,
        second
    )

In [ ]:
customer_ids = customers["customer_id"].tolist()

def choose_receiver(sender):

    receiver = random.choice(customer_ids)

    while receiver==sender:

        receiver = random.choice(customer_ids)

    return receiver

In [ ]:
def choose_transaction_type(occupation):

    profile = transaction_profiles[occupation]

    tx = random.choices(

        population=list(profile.keys()),

        weights=list(profile.values()),

        k=1

    )[0]

    return tx

In [ ]:
def choose_merchant(tx_type, city):

    mapping = {

        "Shopping":"Shopping",

        "Food":"Food",

        "Fuel":"Fuel",

        "Bills":"Bills",

        "Healthcare":"Healthcare"

    }

    if tx_type not in mapping:

        return None

    category = mapping[tx_type]

    possible = merchants[

        (merchants["city"]==city)

        &

        (merchants["category"]==category)

    ]

    if len(possible)==0:

        possible = merchants[
            merchants["category"]==category
        ]

    return possible.sample(1).iloc[0]

In [ ]:
def choose_atm(city_id):

    possible = atms[atms["city_id"]==city_id]

    if len(possible)==0:

        possible = atms

    return possible.sample(1).iloc[0]

In [ ]:
def simulate_customer_day(customer, day):

    transactions = []

    # 90% chance that the customer is active today
    if random.random() > 0.9:
        return transactions

    num_transactions = random.randint(
        1,
        customer["avg_transactions_per_day"]
    )

    for _ in range(num_transactions):

        tx_type = choose_transaction_type(
            customer["occupation"]
        )

        amount = generate_amount(
            customer["occupation"],
            tx_type
        )

        timestamp = generate_timestamp(day)

        atm = choose_atm(customer["city"])

        merchant = choose_merchant(
            tx_type,
            customer["city"]
        )

        if merchant is not None:

            receiver = merchant["merchant_id"]
            receiver_name = merchant["merchant_name"]
            is_merchant = True

        else:

            receiver = choose_receiver(
                customer["customer_id"]
            )

            receiver_name = receiver
            is_merchant = False

        transactions.append({

            "sender": customer["customer_id"],

            "receiver": receiver,

            "receiver_name": receiver_name,

            "is_merchant": is_merchant,

            "transaction_type": tx_type,

            "amount": amount,

            "timestamp": timestamp,

            "atm_id": atm["atm_id"],

            "city": atm["city"],

            "state": atm["state"],

            "latitude": atm["latitude"],

            "longitude": atm["longitude"],

            "fraud_type": customer["risk_label"]

        })

    return transactions

In [ ]:
customers.columns

Index(['customer_id', 'name', 'age', 'gender', 'occupation', 'monthly_income',
       'city_id', 'city', 'state', 'country', 'home_latitude',
       'home_longitude', 'preferred_transactions', 'avg_transactions_per_day',
       'risk_label'],
      dtype='object')

In [ ]:
def simulate_day(day):

    all_transactions = []

    for _, customer in customers.iterrows():

        tx = simulate_customer_day(
            customer,
            day
        )

        all_transactions.extend(tx)

    return all_transactions

In [ ]:
all_transactions = []

NUM_DAYS = 7

for day in range(NUM_DAYS):

    print(f"Generating Day {day+1}")

    day_transactions = simulate_day(day)

    all_transactions.extend(day_transactions)

transactions = pd.DataFrame(all_transactions)

Generating Day 1
Generating Day 2
Generating Day 3
Generating Day 4
Generating Day 5
Generating Day 6
Generating Day 7


In [ ]:
transactions.insert(
    0,
    "transaction_id",
    [
        f"TX{i:08d}"
        for i in range(1,len(transactions)+1)
    ]
)

In [ ]:
transactions = transactions.sort_values(
    "timestamp"
).reset_index(drop=True)

In [ ]:
transactions.to_csv(
    f"{PROJECT_PATH}/dataset/transactions.csv",
    index=False
)

print("Dataset Generated Successfully!")

print()

print("Total Transactions :",len(transactions))

transactions.head()

Dataset Generated Successfully!

Total Transactions : 148544


,transaction_id,sender,receiver,receiver_name,is_merchant,transaction_type,amount,timestamp,atm_id,city,state,latitude,longitude,fraud_type
0,TX00007570,CUST01729,MER00193,Bharat Petroleum,True,Fuel,3002,2026-01-01 07:00:03,ATM0172,Delhi,Delhi,28.574159,77.160227,Normal
1,TX00017239,CUST04035,MER00158,Bharat Petroleum,True,Fuel,620,2026-01-01 07:00:06,ATM0111,Patna,Bihar,25.642494,85.148727,Normal
2,TX00010111,CUST02337,CUST01403,CUST01403,False,Transfer,33077,2026-01-01 07:00:08,ATM0068,Bhopal,Madhya Pradesh,23.255729,77.462445,Normal
3,TX00009240,CUST02134,CUST02339,CUST02339,False,Card,6389,2026-01-01 07:00:10,ATM0171,Jaipur,Rajasthan,26.917736,75.807042,Normal
4,TX00009635,CUST02235,CUST04794,CUST04794,False,ATM,3950,2026-01-01 07:00:11,ATM0155,Chandigarh,Chandigarh,30.766373,76.738066,Normal


In [ ]:
transactions.shape

(148544, 14)